# Docker and FastAPI Test

## Connection

first: ``make docker-run``

In [8]:
import httpx
import pandas as pd

API = "http://localhost:8000"
client = httpx.Client(base_url=API, timeout=None)  # sin límite: el batch tarda minutos

client.get("/health").json()


{'status': 'ok', 'model_version': 'churn_lightgbm_classifier_brand64/v63'}

## Rate a player

In [9]:
r = client.get("/score/64/2266405")
r.status_code, r.json()


(200,
 {'run_date': '2026-10-06',
  'model_version': 'churn_lightgbm_classifier_brand64/v63',
  'feature_version': 'cutoff 2026-10-06',
  'brand_id': 64,
  'player_id': 2266405,
  'p_churn_60d': 0.26037735849056604,
  'p_churn_7d': 0.27364864864864863,
  'p_churn_14d': 0.3044982698961938,
  'p_churn_30d': 0.38461538461538464,
  'value_at_risk_30d': 0.0,
  'risk_band': 4,
  'top_drivers': 'active_days_l90d (-0.84); wagered_eur_l7d (+0.42); days_since_last_bet (+0.36)',
  'drift_flag': 0})

## Several players at a table

In [13]:
scores = pd.read_parquet("../data/03_output/player_scores/run_date=2026-10-06/brand_id=64.parquet")
sample = scores.sample(5, random_state=1)["player_id"]

rows = [client.get(f"/score/64/{p}").json() for p in sample]
api = pd.DataFrame(rows)
api[["player_id", "p_churn_60d", "risk_band", "value_at_risk_30d", "top_drivers"]]


,player_id,p_churn_60d,risk_band,value_at_risk_30d,top_drivers
0,19366051,0.856346,9,1.403850,active_days_l90d (+1.51); days_since_first_bet...
1,13299046,0.856346,9,3.080663,active_days_l90d (+1.42); days_since_first_bet...
2,19441725,0.467583,5,1.599919,active_days_l90d (+0.96); days_since_last_bet ...
3,7626182,0.010671,1,11.111135,active_days_l90d (-1.48); wagered_eur_l7d (-0....
4,10839974,0.386570,5,4.098898,active_days_l90d (+0.98); days_since_first_bet...


## Verify that the API returns the same result as the batch.

In [16]:
check = api.merge(scores, on="player_id", suffixes=("_api", "_batch"))
(check["p_churn_60d_api"] - check["p_churn_60d_batch"]).abs().max()   # should be 0


np.float64(0.0)

## Errors

In [18]:
print(client.get("/score/64/1").status_code, client.get("/score/64/1").json()    )             # 404
print(client.post("/score/batch", json={"as_of": "ayer"}).status_code  )                        # 422


404 {'detail': 'player 1 is not active in brand 64'}
422


## Complete batch

In [20]:
r = client.post("/score/batch", json={"as_of": "2026-08-01", "brand_id": 64})
r.json()


{'as_of': '2026-08-01',
 'brand_id': 64,
 'model_version': 'churn_lightgbm_classifier_brand64/v63',
 'n_players': 13996,
 'output_path': 'data/03_output/player_scores/run_date=2026-08-01/brand_id=64.parquet',
 'seconds': 10.8}

## Response time per player

In [21]:
import time
t0 = time.perf_counter()
for p in sample:
    client.get(f"/score/64/{p}")
print(f"{(time.perf_counter() - t0) / len(sample) * 1000:.0f} ms por jugador")


21 ms por jugador
